<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W9D3_agentic_agent_student_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tiny Agent with Tools ?

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

In [ ]:
!pip install -q smolagents[transformers] wikipedia

## 1) Define KB

In [ ]:
#To-Do: you can add your own knowledge base snippets here
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]
print('KB entries:', len(kb_snippets))


KB entries: 5


## 2) Define tools

In [ ]:
from smolagents import Tool, TransformersModel, ToolCallingAgent

class KBLookupTool(Tool):
    #To-Do: you can customize the name and description of the tool here for example:
    name = "kb_lookup_tool"
    description = "Looks up relevant information from a custom knowledge base."
    inputs = {'query': {'type': 'string', 'description': 'The query string for knowledge base lookup.'}}
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in q.split())
        ]
        return ''.join(matches) if matches else "No KB match."


class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {
        'a': {'type': 'number', 'description': 'The first number.'},
        'b': {'type': 'number', 'description': 'The second number.'},
        'op': {'type': 'string', 'description': 'The operation to perform (e.g., "add", "multiply").', 'default': 'add', 'nullable': True}
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            return str(a * b)
        return str(a + b)


kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()

## 3) Model (tiny local)

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = TransformersModel(
    #To-Do: set up the model parameters as needed
)

print("Model ready:", MODEL_ID)

/usr/local/lib/python3.12/dist-packages/smolagents/models.py:933: FutureWarning: The 'model_id' parameter will be required in version 2.0.0. Please update your code to pass this parameter to avoid future errors. For now, it defaults to 'HuggingFaceTB/SmolLM2-1.7B-Instruct'.
  warnings.warn(


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Model ready: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 4) Agent

In [ ]:
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=2,
    instructions=(
        "You are an agentic AI that uses tools to answer questions. "
        "Keep answers concise (2-4 sentences). Always cite where evidence came from to stay transparent."
    ),
)

print(agent)

## 5) Test queries

In [ ]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("---")
    print("Q:", q)
    result = agent(q)
    print("Answer:", result)

---
Q: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You're a helpful agent named 'None'.                                                                            │
│ You have been submitted this task by your manager.                                                              │
│ ---                                                                                                             │
│ Task:                                                                                                           │
│ Add 12 and 30.                                                                                                  │
│ ---                                                                                                             │
│ You're helping your manager solve a wider task: so make sure to not provide a one-line answer, but give as much │
│ information as possible to give them a clear understanding of the answer.                                       │
│                                                                                                                 │
│ Your final_answer WILL HAVE to contain these parts:                                                             │
│ ### 1. Task outcome (short version):                                                                            │
│ ### 2. Task outcome (extremely detailed version):                                                               │
│ ### 3. Additional context (if relevant):                                                                        │
│                                                                                                                 │
│ Put all these in your final_answer tool, everything that you do not pass as an argument to final_answer will be │
│ lost.                                                                                                           │
│ And even if your task resolution is not successful, please return as much context as possible, so that your     │
│ manager can act upon this feedback.                                                                             │
│                                                                                                                 │
╰─ TransformersModel - HuggingFaceTB/SmolLM2-1.7B-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error while parsing tool call from model output: The model output does not contain any JSON blob.

[Step 1: Duration 8.62 seconds| Input tokens: 1,303 | Output tokens: 166]

...

Error while parsing tool call from model output: The model output does not contain any JSON blob.

[Step 2: Duration 15.16 seconds| Input tokens: 3,011 | Output tokens: 692]

Reached max steps.

[Step 3: Duration 21.48 seconds| Input tokens: 4,258 | Output tokens: 1,274]

Answer: Here is the final answer from your managed agent 'None':
### 1. Task outcome (short version):
An agentic AI loop refers to a self-reinforcing cycle where an AI agent continues to perform a task or set of tasks based on its own decision-making processes, without external intervention or oversight. This loop can lead to a continuous repetition of the same actions, often resulting in a lack of progress or improvement in the task's outcome.

### 2. Task outcome (extremely detailed version):
An agentic AI loop can occur when an AI system is designed to operate autonomously, with minimal human intervention or oversight. The AI agent may be programmed to follow a set of rules or algorithms that guide its decision-making processes. However, if these rules or algorithms are flawed or incomplete, the AI agent may continue to perform the same tasks or actions, even if they are not optimal or effective.

### 3. Additional context (if relevant):
Agentic AI loops can be problematic because t